# [INFO] Glu-Stock: 03_EXECUTION_ENGINE
**Phase**: Institutional Risk Management & Portfolio Execution (v18.1)

This notebook acts as the 'Closer' and 'Executor'. It manages open trades (Stop-Loss/Take-Profit) and opens new positions using ATR-based sizing.

In [ ]:
# [INSTALL] SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas python-dotenv ta


In [ ]:
# [INFO] SECTION 2: INFRASTRUCTURE (Firebase, Secrets & Helpers)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf, warnings
from firebase_admin import credentials, firestore
from datetime import datetime
warnings.filterwarnings('ignore')

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try:
                raw = user_secrets.get_secret("FIREBASE_KEY_JSON")
                return {"key": json.loads(raw)}
            except Exception as e:
                print(f"[ERROR] FIREBASE_KEY_JSON missing or invalid! Error: {e}")
                return {"key": None}
        else:
            from dotenv import load_dotenv
            load_dotenv()
            raw = os.getenv("FIREBASE_KEY_JSON")
            if not raw: return {"key": None}
            return {"key": json.loads(raw)}

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            if not secrets.get('key'):
                raise ValueError("FIREBASE_KEY_JSON is missing. Check Kaggle Secrets.")
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()

    def wait_for_queue(self, queue_name: str, max_retries=20, interval=60):
        import time
        for i in range(max_retries):
            docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
            if docs:
                tasks = []
                for doc in docs:
                    dt = doc.to_dict()
                    tasks.append(dt.get('payload', dt))
                    doc.reference.delete()
                return tasks
            if i < max_retries - 1:
                print(f"[WAIT] {queue_name} queue empty. Retrying ({i+1}/{max_retries}) in {interval}s...", flush=True)
                time.sleep(interval)
        return []
        
    def get_active_trades(self):
        docs = self.db.collection("glu_stock_trades").where("status", "==", "OPEN").get()
        return [(doc.id, doc.to_dict()) for doc in docs]
        
    def update_trade(self, doc_id, data):
        self.db.collection("glu_stock_trades").document(doc_id).update(data)
        
    def insert_trade(self, trade_data):
        self.db.collection("glu_stock_trades").add(trade_data)
        
    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({
            'timestamp': datetime.now().isoformat(), 
            'phase': phase.upper(), 
            'details': details
        })


In [ ]:
# [BRAIN] SECTION 3: CORE LOGIC (ATR Sizing & Trade Closer)
import ta

class RiskManager:
    def __init__(self, equity_risk=0.01, portfolio_equity=100000000):
        self.equity_risk = equity_risk # Risk 1% of total capital per trade
        self.portfolio_equity = portfolio_equity
        
    def calculate_position(self, df, ticker):
        try:
            close = df['Close'].squeeze()
            high = df['High'].squeeze()
            low = df['Low'].squeeze()
            
            curr_price = float(close.iloc[-1])
            atr = ta.volatility.AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range().iloc[-1]
            
            # Stop Loss at 2.0 * ATR below price
            stop_loss = curr_price - (2.0 * atr)
            risk_per_share = curr_price - stop_loss
            
            # Capital risk amount (e.g. 1% of 100M = 1M)
            total_risk_amt = self.portfolio_equity * self.equity_risk
            
            shares = int(total_risk_amt / (risk_per_share + 1e-7))
            return shares, stop_loss, curr_price + (3.0 * atr) # Target at 3.0 * ATR
        except:
            return 0, 0 ,0

class TradingAgent:
    def __init__(self, fb):
        self.fb = fb
        self.risk = RiskManager()

    def manage_open_trades(self):
        active = self.fb.get_active_trades()
        if not active: return
        
        print(f'[INFO] Managing {len(active)} active trades...')
        for doc_id, t in active:
            try:
                ticker = t['ticker']
                df = yf.download(ticker, period='2d', progress=False)
                curr_price = float(df['Close'].iloc[-1])
                
                # Exit check
                if curr_price <= t['stop_loss']:
                    reason = "STOP LOSS"
                elif curr_price >= t['take_profit']:
                    reason = "TAKE PROFIT"
                else:
                    continue
                
                # Close position
                pnl = (curr_price - t['entry_price']) * t['shares']
                self.fb.update_trade(doc_id, {
                    'status': 'CLOSED', 
                    'exit_price': curr_price, 
                    'exit_date': datetime.now().isoformat(),
                    'reason': reason,
                    'pnl': pnl
                })
                self.fb.log_event('CLOSER', f'Closed {ticker} @ {curr_price} ({reason}) | PnL: {pnl:,.0f}')
                print(f'[EXIT] {ticker} closed at {curr_price} ({reason})')
            except Exception as e: print(f'[WARN] Closer error for {ticker}: {e}')

    def execute_signals(self, signals):
        active_count = len(self.fb.get_active_trades())
        if active_count >= 10:
            print('[BLOCK] Portfolio full (10 positions). Ignoring new signals.')
            return
            
        for ticker, data in signals.items():
            try:
                df = yf.download(ticker, period='30d', progress=False)
                shares, sl, tp = self.risk.calculate_position(df, ticker)
                
                if shares > 0:
                    trade_data = {
                        'ticker': ticker,
                        'shares': shares,
                        'entry_price': data['price'],
                        'stop_loss': sl,
                        'take_profit': tp,
                        'entry_date': datetime.now().isoformat(),
                        'conviction': data.get('cnn_confidence', 0),
                        'status': 'OPEN'
                    }
                    self.fb.insert_trade(trade_data)
                    self.fb.log_event('EXECUTION', f'Opened {ticker} @ {data["price"]} | SL: {sl:,.0f} | TP: {tp:,.0f}')
                    print(f'[BUY] {ticker} opened for {shares} shares.')
            except Exception as e: print(f'[WARN] Exec error {ticker}: {e}')


In [ ]:
# [RUN] SECTION 4: MAIN EXECUTION
def run_execution_cycle():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    agent = TradingAgent(fb)
    
    # 1. Manage existing trades (Close if SL/TP hit)
    agent.manage_open_trades()
    
    # 2. Pull new signals from queue
    signal_batches = fb.wait_for_queue('signals', max_retries=10)
    if not signal_batches: 
        print('[EMPTY] No new signals in queue.')
        return
    
    all_signals = {}
    for batch in signal_batches: all_signals.update(batch)
    
    # 3. Open new positions
    agent.execute_signals(all_signals)
    print('[OK] Execution cycle complete.')

run_execution_cycle()